## Provenance, scope, and design decisions

Notebook 3 of the `cbase2/` pipeline: the **financing core** (workplan Stage 1
item 1, Foundation II). It implements and smoke-tests the three financing
closures and retires the unfinanced (manna) autonomous demand of the legacy
pipeline. The design decisions below were fixed **before implementation**
(full rationale in `process_comments.md`, chapter Notebook 03):

- **Common programme frame.** Real bundle $g_i = m\,G_0\,\psi_i$ (numeraire
  units), incidence $\psi_i$ = year-1 (2024) sectoral shares of
  `data_raw/impulses.csv`, scale $G_0 = 40{,}300$ EUR m (the paper's
  stated programme), model multiplier $m$. The same $\psi$ and scale are
  used across F1/F2/F3 for comparability.
- **F1 preference reallocation.** Household preference shifters
  $d_i = 1 + G_0 \psi^{F1}_i / (cs_i E_0)$ within the budget-consistent CES
  normalizer; no additive demand; the composition of $E$ shifts toward
  programme sectors, all other categories shrink pro rata.
- **F2 tax-financed.** Additive bundle $g$; endogenous public budget
  $T(p) = \sum_i p_i g_i$ levied as a lump-sum tax:
  $E = w\sum_i L_i - T$.
- **F3 external debt.** Same bundle; household untaxed; external balance
  $F = \sum_i p_i g_i$ recorded post-solve (accounting open-economy
  closure).
- Unit conversion: 1 model income unit = GDP at basic prices
  (`gdp_production`), so $G_0$ enters as $40{,}300\,/\,\text{GDP}_P$.
- Elasticities follow the parent's calibration convention:
  $\theta = 0.5$, $\epsilon = 0.5$, $\sigma = 0.9$, $\eta = 0.5$.


In [ ]:
import Pkg

# Activate the BeyondHulten project only when the running environment lacks
# CSV (container stack environments skip this).
function find_project(start)
    d = abspath(start)
    while d != dirname(d)
        isfile(joinpath(d, "Project.toml")) && return d
        d = dirname(d)
    end
    return nothing
end
if Base.find_package("CSV") === nothing
    Pkg.activate(find_project(@__DIR__))
end
using CSV, DataFrames, LinearAlgebra, Statistics, NonlinearSolve, Plots

CBROOT = abspath(@__DIR__)                    # cbase2/
OUT    = joinpath(CBROOT, "data_processed")
PLOTS  = joinpath(CBROOT, "plots")

# ── cbase2 model kernel: core copies + the financing module ──
# (include order matters: financing.jl after the core, which defines
#  AbstractFinancing via the recorded surgical edit to interface.jl)
for f in ["interface.jl", "solution.jl", "ces.jl", "mobile_labor.jl",
          "leontief.jl", "util.jl"]
    include(joinpath(CBROOT, "src", "core", f))
end
include(joinpath(CBROOT, "src", "financing.jl"))

N = 71
# read_data resolves pwd()/data/<file>; inside cbase2 `data` links to data_raw.
data = read_data("I-O_DE2019_formatiert.csv")
println("kernel + financing loaded; N = ", N,
        "; gdp_production = ", round(data.gdp_production), " EUR m")


## Step 1 -- Programme calibration: incidence $\psi$, scale, and financing objects

The incidence vector is the **2024 row** of `data_raw/impulses.csv`
(27 impulse years 2024--2050; the paper's ~EUR 40.3 bn anchor and the raw
row-1 total of ~EUR 80.8 bn differ -- see `process_comments.md`). The
impulse file carries 73 data columns: the 71 sectors plus two all-zero
non-sector columns ("Goods ... private households", "Wages and salaries");
the 71-sector alignment against `data_raw/sector_names.txt` is asserted.

F1 can only redirect spending toward categories the household actually
consumes ($cs_i > 0$): the 10 zero-$cs$ sectors (no impulse mass in this
data) are dropped from $\psi^{F1}$ and the remaining weights renormalized.


In [ ]:
# ---------------------------------------------------------------------------
# Step 1 -- Programme incidence ψ (2024 impulse shares) and financing objects.
# ---------------------------------------------------------------------------
imp = CSV.read(joinpath(CBROOT, "data_raw", "impulses.csv"), DataFrame)
@assert nrow(imp) == 27 && :year in propertynames(imp)
row2024 = imp[imp.year .== 2024, :]
@assert nrow(row2024) == 1

# 73 data columns: 71 sectors + 2 non-sector columns (must be all-zero).
sector_names = readlines(joinpath(CBROOT, "data_raw", "sector_names.txt"))
@assert length(sector_names) == N "sector_names.txt must list the 71 sectors"
@assert collect(propertynames(imp))[3:73] == Symbol.(sector_names) "impulse columns must match the sector list"
extra = Matrix{Float64}(row2024[1:1, 74:75])[:]
@assert all(extra .== 0.0) "non-sector impulse columns must be zero"

imp_2024 = Matrix{Float64}(row2024[1:1, 3:73])[:]
ψ = imp_2024 ./ sum(imp_2024)
@assert isapprox(sum(ψ), 1.0; rtol=1e-12) && all(ψ .>= 0)

# Scale: 1 model income unit = GDP at basic prices.
G0_MODEL = 40_300.0 / data.gdp_production
g = G0_MODEL .* ψ
@assert isapprox(sum(g), G0_MODEL; rtol=1e-12)

# F1 incidence restricted to consumable categories (cs_i > 0), renormalized.
cs = data.consumption_share
n_zero_cs = count(<=(0), cs)
dropped = sum(ψ[cs .<= 0])
ψF1 = copy(ψ); ψF1[cs .<= 0] .= 0.0
ψF1 ./= sum(ψF1)
d = 1.0 .+ G0_MODEL .* ψF1 ./ max.(cs, eps())

println("ψ (2024 impulse shares): sum = 1, top-5 shares = ",
        round.(sort(ψ; rev=true)[1:5]; digits=4))
println("G0 in model units (m = 1) = ", round(G0_MODEL; digits=6),
        "  (= 40,300 / ", round(data.gdp_production), " EUR m)")
println("zero-cs sectors: ", n_zero_cs, " | impulse mass dropped from ψF1: ",
        round(dropped; digits=8))
println("F1 shift d: min = ", round(minimum(d); digits=4),
        ", max = ", round(maximum(d); digits=4), " (sector ", argmax(d), ")")


## Step 2 -- Reference solve (baseline integrity)

The no-financing reference under the mobile closure must reproduce the
parent pipeline's reference real GDP ($\approx 0.9998546537$) exactly --
the financing hook is a no-op at `NoFinancing()`, so any deviation would
mean the surgical core edit changed the baseline system.


In [ ]:
# ---------------------------------------------------------------------------
# Step 2 -- Reference solve (NoFinancing, :mobile).
# ---------------------------------------------------------------------------
shocks = Shocks(ones(N), ones(N), zeros(N))       # legacy extra-demand fields stay ZERO
ref = mobile_labor_model(data, shocks, 0.5, 0.5, 0.9, 0.5)
ref_sol = solve(ref)
init_ref = [ref_sol.prices_raw; ref_sol.quantities; ref_sol.wages_raw[1]]

@assert isapprox(real_gdp(ref_sol), 0.9998546537; atol=1e-7) "reference real GDP must reproduce the parent value"
println("reference real GDP = ", real_gdp(ref_sol), "  (parent: 0.9998546537) -- OK")


## Step 3 -- Headline check: F1 / F2 / F3 at $m = 1$, mobile closure

Every solve is checked for (i) equilibrium residuals, (ii) the exact
household budget identity $\sum_i p_i c_i = E$, (iii) the financing records
($T = \sum p_i g_i$ under F2; $F = \sum p_i g_i$ under F3). Headline set:
Tornqvist real GDP (B&F consumption metric), employment, external balance.
A supplementary **total-final-demand index** (consumption + public bundle)
is recorded alongside the consumption metric: under tax financing the two
diverge by construction (the tax contracts $c$ while $g$ expands), and the
assessment requires deviations from pre-registered signatures to be
reported as findings, not hidden by metric choice.


In [ ]:
# ---------------------------------------------------------------------------
# Step 3 -- F1/F2/F3 solves at m = 1 (:mobile) with budget/residual checks.
# ---------------------------------------------------------------------------
function check_and_headline(tag, model, sol)
    p, q, w = sol.prices_raw, sol.quantities, sol.wages_raw[1]
    fixed = labor_closure(model) isa FixedWageClosure
    X = fixed ? [p; q] : [p; q; w]
    rmax = maximum(abs, equilibrium_residuals(model, X))
    L = sum(sectoral_labor_demand(p, q, w, model))
    T = public_budget(model.financing, p)
    F = external_balance(model.financing, p)
    E = household_expenditure(model.financing, w * L, p)
    bud = dot(p, sol.consumption) - E
    @assert rmax < 1e-6      "$tag: equilibrium residuals too large ($rmax)"
    @assert abs(bud) < 1e-9  "$tag: household budget identity violated ($bud)"
    @assert E > 0            "$tag: household expenditure base must be positive at equilibrium"
    # Supplementary total-final-demand Tornqvist index (c + g) vs baseline c.
    base_c = data.consumption_share .* sum(data.labor_share)
    base_total = base_c
    total_vec = sol.consumption .+ additive_demand(model.financing, N)
    tfd = tornqvist_quantity_index(p, total_vec, ones(N), base_total)
    (tag = tag, resid = rmax, real_gdp = real_gdp(sol), tfd = tfd,
     employment = L, T = T, F = F, budget_gap = bud)
end

fin1 = PreferenceReallocation(d)
fin2 = TaxFinanced(g)
fin3 = ExternalDebt(g)

results = DataFrame()
for (tag, fin) in [("F1_preference_reallocation", fin1), ("F2_tax_financed", fin2), ("F3_external_debt", fin3)]
    model = mobile_labor_model(data, shocks, 0.5, 0.5, 0.9, 0.5; financing=fin)
    push!(results, check_and_headline(tag * " mobile", model, solve(model; init=init_ref)))
end
results


## Step 4 -- Sticky-wage smoke tests (GAMMA row) and the $\eta = 1$ guard

Under `:fixed`, F2 must solve (the additive bundle anchors scale) with the
same budget checks. At $\eta = 1$ the fixed-wage system is homogeneous and
scale-indeterminate; without an additive anchor it must throw the
documented guard. The pre-registered GAMMA $\times$ F2 signature ("the
extensive margin: employment absorbs the shock") derives from the retired
UNFINANCED run (+19.3 pp); under tax financing the lump-sum tax contracts
household demand while the bundle expands public demand, so an
employment-neutral outcome is admissible and will be reported against the
pre-registration as a deviation finding (Stage 2).


In [ ]:
# ---------------------------------------------------------------------------
# Step 4 -- :fixed closure smoke tests + η=1 scale-indeterminacy guard.
# ---------------------------------------------------------------------------
mfx2 = mobile_labor_model(data, shocks, 0.5, 0.5, 0.9, 0.5; closure=:fixed, financing=fin2)
sol_fx2 = solve(mfx2; init=[ref_sol.prices_raw; ref_sol.quantities])
push!(results, check_and_headline("F2_tax_financed fixed", mfx2, sol_fx2))

# η = 1 + NoFinancing: must throw the scale-indeterminacy guard.
mfx1 = mobile_labor_model(data, shocks, 0.5, 0.5, 0.9, 1.0; closure=:fixed)
guard_fired = try
    solve(mfx1); false
catch e
    e isa ArgumentError || rethrow()
    println("η=1 fixed without additive anchor: guard fired as designed")
    true
end
@assert guard_fired "the η=1 scale-indeterminacy guard must fire without an additive anchor"
results


## Step 5 -- Emit financing calibration artifacts

Two CSVs under `data_processed/`: the programme calibration (incidence
vectors, shifters, bundle, constants) and the smoke-test headline table.
The incidence figure renders inline next to its computation and is saved
to `plots/`.


In [ ]:
# ---------------------------------------------------------------------------
# Step 5 -- Persist calibration + smoke-test artifacts; incidence figure.
# ---------------------------------------------------------------------------
calib = DataFrame(
    sector  = 1:N,
    psi_2024 = ψ,
    psi_F1   = ψF1,
    d_F1     = d,
    g_bundle = g,
    consumption_share = cs)
CSV.write(joinpath(OUT, "financing_calibration.csv"), calib)

CSV.write(joinpath(OUT, "financing_smoke_results.csv"), results)

top = min(15, N)
p = bar(1:top, ψ[1:top]; label = "",
        title = "Programme incidence ψ (2024 impulse shares, top 15 sectors)",
        xlabel = "sector", ylabel = "share",
        size = (900, 400), margin = 4Plots.mm)
savefig(p, joinpath(PLOTS, "03_programme_incidence.png"))
display(p)
println("Saved plots/03_programme_incidence.png")

for f in ["financing_calibration.csv", "financing_smoke_results.csv"]
    @assert isfile(joinpath(OUT, f)) "missing artifact: $f"
    println("  wrote data_processed/", f)
end


## Run summary

Gate for notebook 03: reference integrity, all residual/budget checks, the
$\eta=1$ guard, and both artifacts. The 5 x 3 matrix itself is filled in
notebook 07 under the `05` pre-registration record.


In [ ]:
# ---------------------------------------------------------------------------
# Final gate for notebook 03.
# ---------------------------------------------------------------------------
@assert isapprox(real_gdp(ref_sol), 0.9998546537; atol=1e-7) "reference integrity"
@assert all(results.resid .< 1e-6) "all smoke solves must converge to machine-tolerance residuals"
@assert maximum(abs.(results.budget_gap)) .< 1e-9 "household budget identity for every cell"
@assert guard_fired "η=1 guard"
println("All 03 assertions passed. Cells smoke-tested:")
println(results[:, [:tag, :real_gdp, :tfd, :employment, :T, :F]])
